# LoRA fine-tuning

**Purpose**: Fine-tune Llama-3.1-8B-Instruct with LoRA on K random target labels per pair, then run zero-shot inference on the target test set. Compares against Ditto warm-start K=100 (SFT paradigm) and LLM K=2 random-stratified (LLM prompting paradigm) reported in paper.

**Design**:
- Base model: `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` (pre-quantized, Colab-friendly, no gating)
- LoRA rank 16, alpha 32, dropout 0.05
- Target-only fine-tuning (no source pre-training)
- K ∈ {25, 100} target labels sampled uniformly at random per (pair, seed)
- 3 seeds (42, 123, 456)
- 4 clean pairs (P1–P4)
- 3 epochs per fine-tune
- Inference: same prompt format as training, zero demos in the prompt

**Total cells**: 4 pairs × 3 seeds × 2 K values = 24 fine-tune runs, each producing one `metrics.json`.

**Runtime estimate**:
- T4 GPU (free Colab): ~20 min per K=25 cell, ~40 min per K=100 cell → total ~12 hours
- A100 (Colab Pro+): ~5–10 min per cell → total ~2–3 hours

**Costs**: no API spend (all local Colab GPU); ~1 Colab Pro session or ~12 hours on free T4.

**Output**: `results/runs/main_matrix/llm-lora/<pair>/k_<K>_seed_<seed>/metrics.json` on the mounted Drive.

## 1. Bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys

BASE = os.environ.get('REPO_ROOT') or '/content/drive/MyDrive/cd-er-paradigm-choice'
assert os.path.exists(BASE), f'BASE not found: {BASE} — check your Drive layout'
sys.path.insert(0, BASE)
os.chdir(BASE)
print(f'BASE = {BASE}')

## 2. Install dependencies

Unsloth handles the messy setup of transformers + peft + bitsandbytes for 4-bit LoRA on Colab. Trl provides `SFTTrainer` for supervised fine-tuning.

In [ ]:
%pip install -q 'unsloth @ git+https://github.com/unslothai/unsloth.git' 'trl<0.9.0' 'peft<0.13.0' 'accelerate>=0.30.0' 'bitsandbytes>=0.43.0' datasets

import torch
assert torch.cuda.is_available(), 'GPU not attached — Runtime → Change runtime type → T4 or A100'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Config

In [ ]:
# ---------------------------------------------------------------------------
# What to run
# ---------------------------------------------------------------------------
PAIRS = {
    'P1_WA_AB': ('Structured/Walmart-Amazon',      'Textual/Abt-Buy'),
    'P2_Co_Wa': ('wdc/computers',                    'wdc/watches'),
    'P3_WA_DA': ('Structured/Walmart-Amazon',      'Structured/DBLP-ACM'),
    'P4_WA_AG': ('Structured/Walmart-Amazon',      'Structured/Amazon-Google'),
}
K_VALUES = [25, 100]                # target-label budgets to test
SEEDS    = [42, 123, 456]

# ---------------------------------------------------------------------------
# Model + LoRA hyperparameters
# ---------------------------------------------------------------------------
MODEL_NAME  = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 1024                  # covers ~800-token Ditto-format prompts with headroom
LOAD_IN_4BIT = True

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj']

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
NUM_EPOCHS     = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 4                  # effective batch size = 8
LR             = 2e-4
WARMUP_STEPS   = 5
WEIGHT_DECAY   = 0.01
OPTIM          = 'adamw_8bit'       # fits T4 memory budget

# ---------------------------------------------------------------------------
# Data paths (Magellan datasets shipped in the repo's ditto/ subtree)
# ---------------------------------------------------------------------------
from pathlib import Path
MAGELLAN_ROOT = Path(BASE) / 'ditto' / 'data' / 'er_magellan'
assert MAGELLAN_ROOT.exists(), f'Magellan data missing at {MAGELLAN_ROOT}'

# ---------------------------------------------------------------------------
# Output directory
# ---------------------------------------------------------------------------
RUNS_ROOT = Path(BASE) / 'results' / 'runs' / 'main_matrix' / 'llm-lora'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

SKIP_IF_DONE = True                 # skip cells whose metrics.json already exists

print(f'Config: {len(PAIRS)} pairs × {len(K_VALUES)} K × {len(SEEDS)} seeds = {len(PAIRS)*len(K_VALUES)*len(SEEDS)} fine-tune runs')
print(f'Base model: {MODEL_NAME}')
print(f'Output: {RUNS_ROOT}')

## 4. Data loading utilities

Ditto's format is `<record_A>\t<record_B>\t<label>` per line, records serialised as `COL <attr> VAL <value> ...`. We convert each pair into a chat-completion instruction:
- **User turn**: same prompt template used in the LLM-paradigm main matrix runner (see Sec. 3.4), with the two demo slots omitted (this is a zero-shot LLM at inference time after fine-tuning).
- **Assistant turn**: the ground-truth label as `Yes` or `No`.

This aligns the training format with the inference format so fine-tuning teaches the model *what to output*, not just what to match.

In [ ]:
import random

# Domain name for each pair (matches the main matrix's DOMAIN_INFO for prompt consistency)
DOMAIN_INFO = {
    'Structured/Walmart-Amazon': 'product',
    'Structured/Amazon-Google':  'product',
    'Textual/Abt-Buy':           'product',
    'wdc/computers':             'product',
    'wdc/watches':               'product',
    'Structured/DBLP-ACM':       'publication',
}


def read_ditto_split(magellan_dir: Path):
    """Return list of (left_record, right_record, label:int) for train/valid/test."""
    out = {}
    for split in ('train', 'valid', 'test'):
        rows = []
        with (magellan_dir / f'{split}.txt').open() as f:
            for line in f:
                parts = line.rstrip('\n').split('\t')
                if len(parts) == 3:
                    rows.append((parts[0], parts[1], int(parts[2])))
        out[split] = rows
    return out


def build_prompt(left: str, right: str, target_domain: str) -> str:
    """Same prompt template as main matrix build_cider_prompt with zero demonstrations."""
    d = target_domain
    D = d.capitalize()
    lines = [
        f'Do the two following {d} descriptions refer to the same {d}?',
        f'{D}1: {left}',
        f'{D}2: {right}',
        'Answer with Yes or No only.',
    ]
    return '\n'.join(lines)


def to_chat_example(left, right, label, target_domain, tokenizer):
    """Format one training pair as a chat message and render with the tokenizer's template."""
    prompt = build_prompt(left, right, target_domain)
    answer = 'Yes' if label == 1 else 'No'
    messages = [
        {'role': 'user',      'content': prompt},
        {'role': 'assistant', 'content': answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def sample_k_labels(train_pairs, k: int, seed: int):
    """Uniform random K-label sample. Matches Ditto's warm-start K=100 protocol."""
    rng = random.Random(seed)
    return rng.sample(train_pairs, min(k, len(train_pairs)))


print('[ok] data utilities loaded')

In [ ]:
%pip install -q --upgrade unsloth_zoo


## 5. Load base model + attach LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = MODEL_NAME,
    max_seq_length    = MAX_SEQ_LEN,
    dtype             = None,          # auto-detect from GPU
    load_in_4bit      = LOAD_IN_4BIT,
)

# Attach LoRA adapters — will be reset per (pair, K, seed) cell in the training loop
model = FastLanguageModel.get_peft_model(
    model,
    r                    = LORA_R,
    target_modules       = LORA_TARGETS,
    lora_alpha           = LORA_ALPHA,
    lora_dropout         = LORA_DROPOUT,
    bias                 = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state         = 42,
)

print(f'[ok] model loaded — {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

## 6. Per-cell runner

For each (pair, K, seed) triple:
1. Sample K training pairs from the target training set (`train.txt`)
2. Fine-tune with LoRA for 3 epochs
3. Run inference on the target test set (`test.txt`)
4. Compute F1, save `metrics.json`

In [ ]:
import json, time, re
from datasets import Dataset
from trl import SFTTrainer, SFTConfig


def parse_yes_no(text: str) -> int:
    """Return 1 if the model answered 'yes'/'match'/'same', 0 otherwise. Matches the main matrix's parser."""
    t = text.lower().strip()
    if t.startswith(('yes', 'match', 'same')):
        return 1
    if t.startswith('no'):
        return 0
    return 0     # ambiguous → 'no'


def compute_f1(preds, labels):
    tp = sum(1 for p, y in zip(preds, labels) if p == 1 and y == 1)
    fp = sum(1 for p, y in zip(preds, labels) if p == 1 and y == 0)
    fn = sum(1 for p, y in zip(preds, labels) if p == 0 and y == 1)
    if 2 * tp + fp + fn == 0:
        return 0.0, 0.0, 0.0
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return f1, p, r


def run_cell(pair_key, target_ds, k, seed):
    out_dir = RUNS_ROOT / pair_key / f'k_{k}_seed_{seed}'
    metrics_path = out_dir / 'metrics.json'

    if SKIP_IF_DONE and metrics_path.exists():
        existing = json.loads(metrics_path.read_text())
        if existing.get('test_f1') is not None:
            print(f'  [skip] {pair_key} K={k} seed={seed}  F1={existing["test_f1"]:.4f}')
            return existing

    out_dir.mkdir(parents=True, exist_ok=True)
    target_dir = MAGELLAN_ROOT / target_ds
    splits = read_ditto_split(target_dir)
    domain = DOMAIN_INFO.get(target_ds, 'entity')

    print(f'\n=== {pair_key} / {target_ds} / K={k} / seed={seed} ===')

    # ---------- Sample K training pairs ----------
    train_sample = sample_k_labels(splits['train'], k, seed)
    train_texts  = [to_chat_example(l, r, y, domain, tokenizer) for l, r, y in train_sample]
    train_ds     = Dataset.from_dict({'text': train_texts})
    print(f'  [train] K={len(train_texts)} pairs sampled')

    # ---------- Reset LoRA weights for this cell ----------
    # Reinitialise LoRA to zero + freshly random by re-attaching
    # (Unsloth's `get_peft_model` above already gave us the adapter graph;
    #  we just need to zero the weights at cell start.)
    for name, param in model.named_parameters():
        if 'lora_' in name and param.requires_grad:
            torch.nn.init.zeros_(param) if 'lora_B' in name else torch.nn.init.kaiming_uniform_(param, a=5**0.5)

    # ---------- Fine-tune ----------
    sft_config = SFTConfig(
        output_dir                  = str(out_dir / 'trainer'),
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = NUM_EPOCHS,
        learning_rate               = LR,
        warmup_steps                = WARMUP_STEPS,
        weight_decay                = WEIGHT_DECAY,
        optim                       = OPTIM,
        logging_steps               = 5,
        save_strategy               = 'no',
        seed                        = seed,
        report_to                   = 'none',
        max_seq_length              = MAX_SEQ_LEN,
        dataset_text_field          = 'text',
    )
    trainer = SFTTrainer(
        model         = model,
        tokenizer     = tokenizer,
        train_dataset = train_ds,
        args          = sft_config,
    )
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    print(f'  [train] done in {train_time:.0f}s')

    # ---------- Inference on target test set ----------
    FastLanguageModel.for_inference(model)
    preds, labels = [], []
    t0 = time.time()
    for i, (left, right, y) in enumerate(splits['test']):
        prompt = build_prompt(left, right, domain)
        messages = [{'role': 'user', 'content': prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
        ).to('cuda')
        with torch.inference_mode():
            out = model.generate(
                input_ids      = inputs,
                max_new_tokens = 5,
                temperature    = 0.0,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
        preds.append(parse_yes_no(gen))
        labels.append(y)
        if (i + 1) % 200 == 0:
            f1_running, _, _ = compute_f1(preds, labels)
            print(f'    [{i+1}/{len(splits["test"])}] running F1={f1_running:.4f}')
    infer_time = time.time() - t0

    f1, p, r = compute_f1(preds, labels)
    print(f'  [done] F1={f1:.4f}, P={p:.4f}, R={r:.4f}, infer={infer_time:.0f}s')

    # ---------- Persist ----------
    with (out_dir / 'predictions.jsonl').open('w') as f:
        for i, (pred, lab) in enumerate(zip(preds, labels)):
            f.write(json.dumps({'i': i, 'pred': pred, 'label': lab}) + '\n')
    metrics = {
        'method': 'llama-3.1-8b-lora',
        'pair_key': pair_key,
        'target_dataset': target_ds,
        'k': k,
        'seed': seed,
        'n_train_pairs': len(train_texts),
        'n_test_pairs': len(splits['test']),
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'num_epochs': NUM_EPOCHS,
        'lr': LR,
        'test_f1': f1,
        'precision': p,
        'recall': r,
        'train_time_sec': train_time,
        'infer_time_sec': infer_time,
    }
    metrics_path.write_text(json.dumps(metrics, indent=2))
    return metrics


print('[ok] runner ready')

## 7. Main loop

In [ ]:
results = []
for pair_key, (source_ds, target_ds) in PAIRS.items():
    for k in K_VALUES:
        for seed in SEEDS:
            try:
                r = run_cell(pair_key, target_ds, k, seed)
                results.append(r)
            except Exception as e:
                print(f'[ERR] {pair_key} K={k} seed={seed}: {e}')
                import traceback; traceback.print_exc()

print(f'\n[complete] {len(results)} cells produced results')

## 8. Aggregate per (pair, K): three-seed mean ± std

In [ ]:
import json, statistics
from collections import defaultdict

by_cell = defaultdict(list)
for f in sorted(RUNS_ROOT.rglob('metrics.json')):
    d = json.loads(f.read_text())
    if d.get('test_f1') is not None:
        by_cell[(d['pair_key'], d['k'])].append(d['test_f1'])

print(f'{"pair":<12} {"K":>5} {"mean F1":>10} {"std":>8} {"seeds":>7}')
print('-' * 50)
for pk in ['P1_WA_AB', 'P2_Co_Wa', 'P3_WA_DA', 'P4_WA_AG']:
    for k in K_VALUES:
        vals = by_cell.get((pk, k), [])
        if vals:
            mean = statistics.mean(vals)
            std  = statistics.stdev(vals) if len(vals) > 1 else 0.0
            print(f'{pk:<12} {k:>5} {mean:>10.4f} {std:>8.4f} {len(vals):>7}')
        else:
            print(f'{pk:<12} {k:>5} {"—":>10} {"—":>8} {0:>7}')

## 9. Push results to git

In [ ]:
!git add results/runs/main_matrix/llm-lora/
!git status --short
!git commit -m 'LoRA fine-tuning LLM LoRA fine-tune: K in {25,100} × 4 pairs × 3 seeds' || echo 'nothing to commit'
!git push